# Exercise 4.1: Routing Features from OpenRouteService in Mainz

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/04_spatial_data_analysis/notebooks/exercise_4_1_openrouteservice_routing_mainz.ipynb)

This feature-acquisition exercise uses OpenRouteService to calculate route distance, duration, and geometry features in Mainz. These routing features can enrich downstream geospatial learning tasks.

## Learning Outcomes

- Call the OpenRouteService directions API to acquire route features.
- Compare walking and cycling routes between Mainz locations.
- Inspect route distance and duration as model-ready contextual features.
- Visualize route geometries as spatial feature context on a Folium map.
- Run simple plausibility tests for route results.

## 1. Colab Setup

In [ ]:
!pip -q install requests folium shapely geopandas pandas


## 2. Imports and Known Mainz Places

In [ ]:
from __future__ import annotations

import getpass
import json
from typing import Any

import folium
import requests
from IPython.display import Markdown, display
from shapely.geometry import shape

MAINZ_CENTER = (49.9929, 8.2473)
KNOWN_PLACES = {
    "mainz_hbf": {"label": "Mainz Hauptbahnhof", "lat": 50.0010, "lon": 8.2587},
    "jgu": {"label": "Johannes Gutenberg University Mainz", "lat": 49.9936, "lon": 8.2419},
    "mainz_dom": {"label": "Mainz Cathedral", "lat": 49.9995, "lon": 8.2742},
    "mainz_theater": {"label": "Staatstheater Mainz", "lat": 50.0002, "lon": 8.2714},
}

for place_id, place in KNOWN_PLACES.items():
    print(place_id, "->", place["label"], (place["lat"], place["lon"]))


## 3. OpenRouteService API Key

Create a free developer key at https://openrouteservice.org/dev/#/signup and paste it below. Do not save API keys in the notebook.

In [ ]:
ORS_API_KEY = getpass.getpass("Paste your OpenRouteService API key for this session: ").strip()
if not ORS_API_KEY:
    raise RuntimeError("No ORS API key entered.")
print("ORS API key loaded for this notebook session.")


## 4. Route Request Helper

In [ ]:
ORS_BASE_URL = "https://api.openrouteservice.org/v2/directions"


def run_ors_route(start_place_id: str, end_place_id: str, profile: str = "foot-walking") -> dict[str, Any]:
    if profile not in {"foot-walking", "cycling-regular", "driving-car"}:
        raise ValueError("Unsupported profile.")
    start = KNOWN_PLACES[start_place_id]
    end = KNOWN_PLACES[end_place_id]
    url = f"{ORS_BASE_URL}/{profile}/geojson"
    payload = {"coordinates": [[start["lon"], start["lat"]], [end["lon"], end["lat"]]]}
    response = requests.post(
        url,
        headers={"Authorization": ORS_API_KEY, "Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    if not response.ok:
        print(response.text[:1500])
        response.raise_for_status()
    return response.json()


def route_summary(route_json: dict[str, Any]) -> dict[str, Any]:
    feature = route_json["features"][0]
    summary = feature["properties"].get("summary", {})
    return {
        "distance_m": float(summary.get("distance", 0)),
        "duration_s": float(summary.get("duration", 0)),
        "geometry": shape(feature["geometry"]),
    }


## 5. Example Route: Mainz Hbf to JGU

In [ ]:
route_json = run_ors_route("mainz_hbf", "jgu", profile="foot-walking")
summary = route_summary(route_json)
print(f"Distance: {summary['distance_m'] / 1000:.2f} km")
print(f"Duration: {summary['duration_s'] / 60:.1f} minutes")
print("Raw route feature keys:", route_json["features"][0].keys())


## 6. Map the Route

In [ ]:
def display_route_map(route_json: dict[str, Any], start_place_id: str, end_place_id: str, title: str) -> folium.Map:
    summary = route_summary(route_json)
    m = folium.Map(location=MAINZ_CENTER, zoom_start=13, tiles="OpenStreetMap", control_scale=True)
    html = f'<div style="position: fixed; top: 10px; left: 50px; z-index: 9999; background: white; padding: 8px 10px; border: 1px solid #999; font-size: 14px;"><strong>{title}</strong></div>'
    m.get_root().html.add_child(folium.Element(html))

    start = KNOWN_PLACES[start_place_id]
    end = KNOWN_PLACES[end_place_id]
    folium.Marker((start["lat"], start["lon"]), popup=start["label"], icon=folium.Icon(color="green")).add_to(m)
    folium.Marker((end["lat"], end["lon"]), popup=end["label"], icon=folium.Icon(color="red")).add_to(m)

    coords = [(lat, lon) for lon, lat in summary["geometry"].coords]
    folium.PolyLine(coords, color="blue", weight=5, opacity=0.8).add_to(m)
    display(Markdown(f"Distance: {summary['distance_m'] / 1000:.2f} km. Duration: {summary['duration_s'] / 60:.1f} minutes."))
    return m


display_route_map(route_json, "mainz_hbf", "jgu", "Walking route: Mainz Hbf to JGU")


## 7. Plausibility Tests

In [ ]:
assert 1500 <= summary["distance_m"] <= 7000, "Distance should be plausible for Mainz Hbf to JGU."
assert summary["duration_s"] > 0, "Duration should be positive."
assert not summary["geometry"].is_empty, "Route geometry should not be empty."
print("Route plausibility checks passed.")


## 8. Student Tasks

1. Compare `foot-walking` and `cycling-regular` between Mainz Hbf and JGU.
2. Calculate a route from Mainz Cathedral to JGU.
3. Add one more known place in Mainz and test a new route.
4. Explain why network distance can differ strongly from straight-line distance.
5. Add a plausibility test for maximum route duration.